In [0]:
from pyspark.sql import functions as F

path_b = "/Volumes/workspace/finance/raw_files/bank_b"

# DIFFERENT: only 4 columns
schema_b = """
    date_raw STRING,
    description_raw STRING,
    amount_raw STRING,
    type_raw STRING
"""

df_b_raw = (spark.read.format("csv")
            .option("header", "true")
            .schema(schema_b)
            .load(path_b))

df_b_raw.printSchema()
display(df_b_raw)

root
 |-- date_raw: string (nullable = true)
 |-- description_raw: string (nullable = true)
 |-- amount_raw: string (nullable = true)
 |-- type_raw: string (nullable = true)



date_raw,description_raw,amount_raw,type_raw
2026-05-01,POS 453212******1234 STARBUCKS COFFEE,395.00,DR
2026-05-03,NETFLIX.COM SUBSCRIPTION,649.00,DR
2026-05-03,POS 453212******1234 AMAZON PAY IN,"1,148.64",DR
2026-05-07,SPOTIFY INDIA,119.00,DR
2026-05-08,DINEOUT*BARBEQUE NATION,"2,696.02",DR
2026-05-12,DINEOUT*BARBEQUE NATION,"2,008.94",DR
2026-05-15,POS 453212******1234 AMAZON PAY IN,642.52,DR
2026-05-16,PAYMENT RECEIVED - THANK YOU,"11,658.11",CR
2026-05-17,POS 453212******1234 MYNTRA DESIGNS,"2,322.27",DR
2026-05-18,POS 453212******1234 STARBUCKS COFFEE,345.00,DR


In [0]:
def to_amount(col_name):
    return F.expr(f"try_cast(regexp_replace(trim({col_name}), ',', '') AS DECIMAL(12,2))")

df_b_step2 = (df_b_raw
    # DIFFERENT: ISO date format
    .withColumn("txn_date", F.expr("to_date(try_to_timestamp(trim(date_raw), 'yyyy-MM-dd'))"))
    # DIFFERENT: one amount column. abs() guarantees it's positive
    .withColumn("amount", F.abs(to_amount("amount_raw"))))

display(df_b_step2.select("date_raw", "txn_date", "amount_raw", "amount"))

date_raw,txn_date,amount_raw,amount
2026-05-01,2026-05-01,395.00,395.00
2026-05-03,2026-05-03,649.00,649.00
2026-05-03,2026-05-03,"1,148.64",1148.64
2026-05-07,2026-05-07,119.00,119.00
2026-05-08,2026-05-08,"2,696.02",2696.02
2026-05-12,2026-05-12,"2,008.94",2008.94
2026-05-15,2026-05-15,642.52,642.52
2026-05-16,2026-05-16,"11,658.11",11658.11
2026-05-17,2026-05-17,"2,322.27",2322.27
2026-05-18,2026-05-18,345.00,345.00


In [0]:
df_b_step3 = df_b_step2.withColumn(
    "direction",
    # DIFFERENT: direction comes from DR/CR, not from which column is filled
    F.when(F.upper(F.trim("type_raw")).isin("DR", "DEBIT"), F.lit("DEBIT"))
     .when(F.upper(F.trim("type_raw")).isin("CR", "CREDIT"), F.lit("CREDIT"))
)

display(df_b_step3.select("type_raw", "direction"))

type_raw,direction
DR,DEBIT
DR,DEBIT
DR,DEBIT
DR,DEBIT
DR,DEBIT
DR,DEBIT
DR,DEBIT
CR,CREDIT
DR,DEBIT
DR,DEBIT


In [0]:
df_b_clean = (df_b_step3
    .withColumn("source_file", F.col("_metadata.file_name"))
    .select(
        "txn_date",
        F.trim("description_raw").alias("description"),
        F.lit(None).cast("string").alias("bank_ref"),   # DIFFERENT: card statements have no ref number
        "amount",
        "direction",
        "source_file",
    ))

df_b_clean.printSchema()
display(df_b_clean)

root
 |-- txn_date: date (nullable = true)
 |-- description: string (nullable = true)
 |-- bank_ref: string (nullable = true)
 |-- amount: decimal(12,2) (nullable = true)
 |-- direction: string (nullable = true)
 |-- source_file: string (nullable = false)



txn_date,description,bank_ref,amount,direction,source_file
2026-05-01,POS 453212******1234 STARBUCKS COFFEE,null,395.00,DEBIT,bank_b_card_2026_05.csv
2026-05-03,NETFLIX.COM SUBSCRIPTION,null,649.00,DEBIT,bank_b_card_2026_05.csv
2026-05-03,POS 453212******1234 AMAZON PAY IN,null,1148.64,DEBIT,bank_b_card_2026_05.csv
2026-05-07,SPOTIFY INDIA,null,119.00,DEBIT,bank_b_card_2026_05.csv
2026-05-08,DINEOUT*BARBEQUE NATION,null,2696.02,DEBIT,bank_b_card_2026_05.csv
2026-05-12,DINEOUT*BARBEQUE NATION,null,2008.94,DEBIT,bank_b_card_2026_05.csv
2026-05-15,POS 453212******1234 AMAZON PAY IN,null,642.52,DEBIT,bank_b_card_2026_05.csv
2026-05-16,PAYMENT RECEIVED - THANK YOU,null,11658.11,CREDIT,bank_b_card_2026_05.csv
2026-05-17,POS 453212******1234 MYNTRA DESIGNS,null,2322.27,DEBIT,bank_b_card_2026_05.csv
2026-05-18,POS 453212******1234 STARBUCKS COFFEE,null,345.00,DEBIT,bank_b_card_2026_05.csv


In [0]:
display(df_b_clean.filter("txn_date IS NULL OR amount IS NULL OR direction IS NULL"))

txn_date,description,bank_ref,amount,direction,source_file
2026-05-15,POS 453212******1234 AMAZON PAY IN,null,null,DEBIT,bank_b_card_2026_05.csv


In [0]:
print(df_a_clean.columns)
print(df_b_clean.columns)
print(df_a_clean.schema == df_b_clean.schema)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6353200852690265>, line 1
----> 1 print(df_a_clean.columns)
      2 print(df_b_clean.columns)
      3 print(df_a_clean.schema == df_b_clean.schema)

NameError: name 'df_a_clean' is not defined